## Run for loop

In [ ]:
import os
import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from datetime import datetime, timedelta

# ==========================================
# 1. PATHS & SETTINGS
# ==========================================
YEAR = "2021"
SITE = "Mozambique_N"

CENTER_LON = 40.52502367602088

CENTER_LAT = -12.930035789266949

# ==========================================
# VIIRS PRODUCT TYPE
# ==========================================
VIIRS_PRODUCT = "_PM_N-"

# Examples:
# "_PM_D-"   -> PM daily
# "_PM_N-"   -> PM night
# "_PM_DY-"  -> PM daytime
# "_AM_D-"   -> AM daily

L3_ROOT = Path(f"../../L3S_STAR/raw")
OSTIA_ROOT = Path(f"../../{SITE}/L4_OSTIA/geotiff")

# Include product name in output folder
PRODUCT_TAG = VIIRS_PRODUCT.strip("_-").replace("-", "")
BASE_ROOT = Path(f"../../{SITE}/result/ready_whole_{YEAR}")

BASE_DATA_DIR = BASE_ROOT / "cloud"

VIIRS_OUT = BASE_DATA_DIR / "viirs"
OISST_OUT = BASE_DATA_DIR / "oisst"
LAND_OUT = BASE_DATA_DIR / "land_mask"
LONLAT_OUT = BASE_DATA_DIR / "lon_lat" / "viirs"

for p in [VIIRS_OUT, OISST_OUT, LAND_OUT, LONLAT_OUT]:
    p.mkdir(parents=True, exist_ok=True)

DST_ROOT = Path(f"{BASE_ROOT.as_posix()}_300")

TARGET_SIZE = 300
CROP_SIZE = 256
HALF_SIZE = CROP_SIZE // 2

N_DEBUG_PLOTS = 3


# ==========================================
# 1.1 GENERATE WHOLE-YEAR DATES
# ==========================================
def generate_all_dates(year):
    start_date = datetime(int(year), 1, 1)
    end_date = datetime(int(year), 12, 31)

    dates = []
    current = start_date

    while current <= end_date:
        dates.append(current.strftime("%Y%m%d"))
        current += timedelta(days=1)

    return dates


DATES = generate_all_dates(YEAR)

print(f"YEAR = {YEAR}")
print(f"Total dates = {len(DATES)}")
print("First date:", DATES[0])
print("Last date :", DATES[-1])
print("VIIRS_PRODUCT:", VIIRS_PRODUCT)
print("BASE_ROOT:", BASE_ROOT)
print("DST_ROOT:", DST_ROOT)


# ==========================================
# 2. FILE SEARCHING HELPERS
# ==========================================
def find_l3_path(date_str):
    yyyy = date_str[:4]
    mm = date_str[4:6]

    pm_dir = L3_ROOT / f"{yyyy}-{mm}" / "PM"

    if not pm_dir.exists():
        return None

    cands = sorted(pm_dir.glob(f"{date_str}*STAR-L3S*.nc"))

    if not cands:
        return None

    preferred = [
        f for f in cands
        if VIIRS_PRODUCT in f.name
    ]

    if preferred:
        return preferred[0]

    print(f"[WARN] {date_str}: no file matched {VIIRS_PRODUCT}; using first available:")
    print("       ", cands[0].name)

    return cands[0]


def find_ostia_path(date_str):
    yyyy = date_str[:4]
    mm = date_str[4:6]

    return OSTIA_ROOT / yyyy / mm / f"{date_str}.tif"


# ==========================================
# 3. SINGLE-DAY PROCESSING
# ==========================================
def process_single_day_with_landmask(date_str, viirs_path, ostia_path):
    print(f"\n🚀 Processing: {date_str}")
    print("VIIRS:", viirs_path)
    print("OSTIA:", ostia_path)

    ds_viirs = xr.open_dataset(viirs_path, decode_timedelta=False)
    ds_viirs = ds_viirs.sortby("lat").sortby("lon")

    sst_viirs = ds_viirs["sea_surface_temperature"].squeeze()

    if float(sst_viirs.max()) > 200:
        sst_viirs = sst_viirs - 273.15

    da_ostia = rioxarray.open_rasterio(ostia_path).squeeze()

    if "band" in da_ostia.coords:
        da_ostia = da_ostia.drop_vars("band")

    if float(da_ostia.max()) > 200:
        da_ostia = da_ostia - 273.15

    da_ostia = da_ostia.where(da_ostia != -9999.0)

    if "x" in da_ostia.dims and "y" in da_ostia.dims:
        da_ostia = da_ostia.rename({"x": "lon", "y": "lat"})

    da_ostia = da_ostia.sortby("lat").sortby("lon")

    lat_idx = np.abs(sst_viirs.lat.values - CENTER_LAT).argmin()
    lon_idx = np.abs(sst_viirs.lon.values - CENTER_LON).argmin()

    start_lat_idx = lat_idx - HALF_SIZE
    end_lat_idx = lat_idx + HALF_SIZE
    start_lon_idx = lon_idx - HALF_SIZE
    end_lon_idx = lon_idx + HALF_SIZE

    if (
        start_lat_idx < 0
        or end_lat_idx > sst_viirs.sizes["lat"]
        or start_lon_idx < 0
        or end_lon_idx > sst_viirs.sizes["lon"]
    ):
        raise ValueError(
            f"Crop outside VIIRS grid for {date_str}. "
            f"lat_idx={lat_idx}, lon_idx={lon_idx}"
        )

    viirs_target = sst_viirs.isel(
        lat=slice(start_lat_idx, end_lat_idx),
        lon=slice(start_lon_idx, end_lon_idx),
    )

    viirs_cropped = viirs_target.values.astype(np.float32)

    lats = viirs_target.lat.values
    lons = viirs_target.lon.values

    ostia_resampled = da_ostia.interp(
        lat=xr.DataArray(lats, dims="lat"),
        lon=xr.DataArray(lons, dims="lon"),
        method="nearest",
    ).values.astype(np.float32)

    finite_ostia = int(np.isfinite(ostia_resampled).sum())

    if finite_ostia == 0:
        raise ValueError(f"OSTIA became all NaN after interpolation for {date_str}")

    # 1 = land / invalid
    # 0 = ocean
    land_mask = np.logical_or(
        np.isnan(ostia_resampled),
        ostia_resampled < 15.0,
    ).astype(np.float32)

    lon_grid, lat_grid = np.meshgrid(lons, lats)
    lonlat_array = np.stack([lon_grid, lat_grid], axis=-1)

    ocean_vals = ostia_resampled[land_mask == 0]

    if ocean_vals.size > 0 and np.any(np.isfinite(ocean_vals)):
        valid_mean = np.nanmean(ocean_vals)
    else:
        valid_mean = 0.0

    ostia_final = np.nan_to_num(ostia_resampled, nan=valid_mean)
    viirs_final = np.nan_to_num(viirs_cropped, nan=0.0)

    file_name = f"{date_str}.npy"

    np.save(VIIRS_OUT / file_name, viirs_final.astype(np.float32))
    np.save(OISST_OUT / file_name, ostia_final.astype(np.float32))
    np.save(LAND_OUT / file_name, land_mask.astype(np.float32))
    np.save(LONLAT_OUT / file_name, lonlat_array.astype(np.float32))

    print(f"✅ Saved raw 256x256 files for {date_str}")
    print("VIIRS shape:", viirs_final.shape)
    print("OSTIA shape:", ostia_final.shape)
    print("LAND shape:", land_mask.shape)
    print("land pixels:", int(np.sum(land_mask == 1)))
    print("ocean pixels:", int(np.sum(land_mask == 0)))

    return date_str


# ==========================================
# 4. PADDING FUNCTIONS
# ==========================================
def pad_array_to_300(arr, pad_value=0.0):
    squeezed = False

    if arr.ndim == 2:
        arr = arr[:, :, None]
        squeezed = True

    h, w, c = arr.shape

    if h == TARGET_SIZE and w == TARGET_SIZE:
        return arr[:, :, 0] if squeezed else arr

    pad_h = TARGET_SIZE - h
    pad_w = TARGET_SIZE - w

    if pad_h < 0 or pad_w < 0:
        raise ValueError(f"Input larger than {TARGET_SIZE}: {arr.shape}")

    top = pad_h // 2
    bottom = pad_h - top
    left = pad_w // 2
    right = pad_w - left

    padded = np.pad(
        arr,
        ((top, bottom), (left, right), (0, 0)),
        mode="constant",
        constant_values=pad_value,
    )

    return padded[:, :, 0] if squeezed else padded


def pad_lonlat_to_300(lonlat):
    h, w, c = lonlat.shape

    if c != 2:
        raise ValueError(f"lonlat must have shape H,W,2. Got {lonlat.shape}")

    lon = lonlat[:, :, 0]
    lat = lonlat[:, :, 1]

    lon_pad = pad_array_to_300(lon, pad_value=0.0)
    lat_pad = pad_array_to_300(lat, pad_value=0.0)

    return np.stack([lon_pad, lat_pad], axis=-1).astype(np.float32)


def save_300_for_date(date_str):
    file_name = f"{date_str}.npy"

    src_viirs = VIIRS_OUT / file_name
    src_ostia = OISST_OUT / file_name
    src_land = LAND_OUT / file_name
    src_lonlat = LONLAT_OUT / file_name

    dst_viirs = DST_ROOT / "cloud" / "viirs"
    dst_ostia = DST_ROOT / "cloud" / "oisst"
    dst_land = DST_ROOT / "cloud" / "land_mask"
    dst_lonlat = DST_ROOT / "cloud" / "lon_lat" / "viirs"

    for p in [dst_viirs, dst_ostia, dst_land, dst_lonlat]:
        p.mkdir(parents=True, exist_ok=True)

    viirs = np.load(src_viirs)
    ostia = np.load(src_ostia)
    land = np.load(src_land)
    lonlat = np.load(src_lonlat)

    viirs_300 = pad_array_to_300(viirs, pad_value=0.0)
    ostia_300 = pad_array_to_300(ostia, pad_value=0.0)
    land_300 = pad_array_to_300(land, pad_value=1.0)
    lonlat_300 = pad_lonlat_to_300(lonlat)

    np.save(dst_viirs / file_name, viirs_300.astype(np.float32))
    np.save(dst_ostia / file_name, ostia_300.astype(np.float32))
    np.save(dst_land / file_name, land_300.astype(np.float32))
    np.save(dst_lonlat / file_name, lonlat_300.astype(np.float32))

    print(f"✅ Saved 300x300 files for {date_str}")

    return viirs_300, ostia_300, land_300


# ==========================================
# 5. PLOT CHECK
# ==========================================
def plot_check(date_str, viirs_300, ostia_300, land_300):
    viirs_vis = np.where(land_300 == 1, np.nan, viirs_300)
    ostia_vis = np.where(land_300 == 1, np.nan, ostia_300)

    vals = np.concatenate([
        viirs_vis[np.isfinite(viirs_vis)].ravel(),
        ostia_vis[np.isfinite(ostia_vis)].ravel(),
    ])

    vmin = np.nanpercentile(vals, 2)
    vmax = np.nanpercentile(vals, 98)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)

    im0 = axes[0].imshow(
        viirs_vis,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        origin="lower",
    )
    axes[0].set_title(f"VIIRS {VIIRS_PRODUCT}\n{date_str}")
    axes[0].axis("off")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(
        ostia_vis,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        origin="lower",
    )
    axes[1].set_title("OSTIA 300")
    axes[1].axis("off")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(
        land_300,
        cmap="gray",
        origin="lower",
    )
    axes[2].set_title("Land Mask 300\nwhite = land/invalid")
    axes[2].axis("off")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


# ==========================================
# 6. EXECUTION: LOOP ALL DAYS
# ==========================================
if __name__ == "__main__":

    success_dates = []
    failed_dates = []

    print("\n==============================")
    print(f"YEAR: {YEAR}")
    print(f"SITE: {SITE}")
    print(f"Processing ALL valid dates")
    print(f"VIIRS_PRODUCT: {VIIRS_PRODUCT}")
    print(f"AOI center: lon={CENTER_LON}, lat={CENTER_LAT}")
    print(f"BASE_ROOT: {BASE_ROOT}")
    print(f"DST_ROOT: {DST_ROOT}")
    print("==============================")

    for target_date in tqdm(DATES, desc=f"Processing all valid days in {YEAR}"):

        viirs_file_path = find_l3_path(target_date)
        ostia_file_path = find_ostia_path(target_date)

        if not viirs_file_path:
            print(f"[MISSING VIIRS] {target_date}")
            failed_dates.append((target_date, "missing_viirs"))
            continue

        if not ostia_file_path.exists():
            print(f"[MISSING OSTIA] {target_date}")
            failed_dates.append((target_date, "missing_ostia"))
            continue

        try:
            generated_date = process_single_day_with_landmask(
                target_date,
                viirs_file_path,
                ostia_file_path,
            )

            viirs_300, ostia_300, land_300 = save_300_for_date(generated_date)

            if len(success_dates) < N_DEBUG_PLOTS:
                plot_check(
                    generated_date,
                    viirs_300,
                    ostia_300,
                    land_300,
                )

            success_dates.append(generated_date)

        except Exception as e:
            print(f"[FAILED] {target_date}: {e}")
            failed_dates.append((target_date, str(e)))
            continue

    print("\n===================================")
    print("FINAL SUMMARY")
    print("===================================")
    print("VIIRS_PRODUCT:", VIIRS_PRODUCT)
    print("Total successful dates:", len(success_dates))
    print("Total failed dates:", len(failed_dates))

    if success_dates:
        print("First 10 successful dates:", success_dates[:10])
        print("Last 10 successful dates:", success_dates[-10:])

    if failed_dates:
        print("First 10 failed dates:", failed_dates[:10])

    DST_ROOT.mkdir(parents=True, exist_ok=True)

    success_txt = DST_ROOT / f"successful_dates_{YEAR}_{PRODUCT_TAG}.txt"
    failed_txt = DST_ROOT / f"failed_dates_{YEAR}_{PRODUCT_TAG}.txt"

    with open(success_txt, "w") as f:
        for d in success_dates:
            f.write(f"{d}\n")

    with open(failed_txt, "w") as f:
        for d, reason in failed_dates:
            f.write(f"{d} : {reason}\n")

    print("\nSaved:")
    print(success_txt)
    print(failed_txt)

    print("\n✅ ALL DONE.")

In [ ]:
Center longitude: 39.730014649575104
Center latitude : -16.644973730979025

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime, timedelta

# ==========================================
# 1. PATHS & SETTINGS
# ==========================================
YEARS = [str(y) for y in range(2020, 2021)]

SITE = "Mozambique_S"

CENTER_LON = 39.730014649575104
CENTER_LAT = -16.644973730979025

# ==========================================
# VIIRS PRODUCT TYPE
# ==========================================
VIIRS_PRODUCT = "_PM_N-"

L3_ROOT = Path("../../L3S_STAR/raw")
OSTIA_ROOT = Path(f"../../{SITE}/L4_OSTIA/geotiff")

PRODUCT_TAG = VIIRS_PRODUCT.strip("_-").replace("-", "")

TARGET_SIZE = 300
CROP_SIZE = 256
HALF_SIZE = CROP_SIZE // 2

N_DEBUG_PLOTS = 3


# ==========================================
# 2. DATE HELPER
# ==========================================
def generate_all_dates(year):
    start_date = datetime(int(year), 1, 1)
    end_date = datetime(int(year), 12, 31)

    dates = []
    current = start_date

    while current <= end_date:
        dates.append(current.strftime("%Y%m%d"))
        current += timedelta(days=1)

    return dates


# ==========================================
# 3. FILE SEARCHING HELPERS
# ==========================================
def find_l3_path(date_str):
    yyyy = date_str[:4]
    mm = date_str[4:6]

    pm_dir = L3_ROOT / f"{yyyy}-{mm}" / "PM"

    if not pm_dir.exists():
        return None

    cands = sorted(pm_dir.glob(f"{date_str}*STAR-L3S*.nc"))

    if not cands:
        return None

    preferred = [f for f in cands if VIIRS_PRODUCT in f.name]

    if preferred:
        return preferred[0]

    print(f"[WARN] {date_str}: no file matched {VIIRS_PRODUCT}; using first available:")
    print("       ", cands[0].name)

    return cands[0]


def find_ostia_path(date_str):
    yyyy = date_str[:4]
    mm = date_str[4:6]

    return OSTIA_ROOT / yyyy / mm / f"{date_str}.tif"


# ==========================================
# 4. SINGLE-DAY PROCESSING
# ==========================================
def process_single_day_with_landmask(
    date_str,
    viirs_path,
    ostia_path,
    VIIRS_OUT,
    OISST_OUT,
    LAND_OUT,
    LONLAT_OUT,
):
    print(f"\n🚀 Processing: {date_str}")
    print("VIIRS:", viirs_path)
    print("OSTIA:", ostia_path)

    ds_viirs = xr.open_dataset(viirs_path, decode_timedelta=False)
    ds_viirs = ds_viirs.sortby("lat").sortby("lon")

    sst_viirs = ds_viirs["sea_surface_temperature"].squeeze()

    if float(sst_viirs.max()) > 200:
        sst_viirs = sst_viirs - 273.15

    da_ostia = rioxarray.open_rasterio(ostia_path).squeeze()

    if "band" in da_ostia.coords:
        da_ostia = da_ostia.drop_vars("band")

    if float(da_ostia.max()) > 200:
        da_ostia = da_ostia - 273.15

    da_ostia = da_ostia.where(da_ostia != -9999.0)

    if "x" in da_ostia.dims and "y" in da_ostia.dims:
        da_ostia = da_ostia.rename({"x": "lon", "y": "lat"})

    da_ostia = da_ostia.sortby("lat").sortby("lon")

    lat_idx = np.abs(sst_viirs.lat.values - CENTER_LAT).argmin()
    lon_idx = np.abs(sst_viirs.lon.values - CENTER_LON).argmin()

    start_lat_idx = lat_idx - HALF_SIZE
    end_lat_idx = lat_idx + HALF_SIZE
    start_lon_idx = lon_idx - HALF_SIZE
    end_lon_idx = lon_idx + HALF_SIZE

    if (
        start_lat_idx < 0
        or end_lat_idx > sst_viirs.sizes["lat"]
        or start_lon_idx < 0
        or end_lon_idx > sst_viirs.sizes["lon"]
    ):
        raise ValueError(
            f"Crop outside VIIRS grid for {date_str}. "
            f"lat_idx={lat_idx}, lon_idx={lon_idx}"
        )

    viirs_target = sst_viirs.isel(
        lat=slice(start_lat_idx, end_lat_idx),
        lon=slice(start_lon_idx, end_lon_idx),
    )

    viirs_cropped = viirs_target.values.astype(np.float32)

    lats = viirs_target.lat.values
    lons = viirs_target.lon.values

    ostia_resampled = da_ostia.interp(
        lat=xr.DataArray(lats, dims="lat"),
        lon=xr.DataArray(lons, dims="lon"),
        method="nearest",
    ).values.astype(np.float32)

    finite_ostia = int(np.isfinite(ostia_resampled).sum())

    if finite_ostia == 0:
        raise ValueError(f"OSTIA became all NaN after interpolation for {date_str}")

    # 1 = land / invalid
    # 0 = ocean
    land_mask = np.logical_or(
        np.isnan(ostia_resampled),
        ostia_resampled < 15.0,
    ).astype(np.float32)

    lon_grid, lat_grid = np.meshgrid(lons, lats)
    lonlat_array = np.stack([lon_grid, lat_grid], axis=-1)

    ocean_vals = ostia_resampled[land_mask == 0]

    if ocean_vals.size > 0 and np.any(np.isfinite(ocean_vals)):
        valid_mean = np.nanmean(ocean_vals)
    else:
        valid_mean = 0.0

    ostia_final = np.nan_to_num(ostia_resampled, nan=valid_mean)
    viirs_final = np.nan_to_num(viirs_cropped, nan=0.0)

    file_name = f"{date_str}.npy"

    np.save(VIIRS_OUT / file_name, viirs_final.astype(np.float32))
    np.save(OISST_OUT / file_name, ostia_final.astype(np.float32))
    np.save(LAND_OUT / file_name, land_mask.astype(np.float32))
    np.save(LONLAT_OUT / file_name, lonlat_array.astype(np.float32))

    print(f"✅ Saved raw 256x256 files for {date_str}")

    return date_str


# ==========================================
# 5. PADDING FUNCTIONS
# ==========================================
def pad_array_to_300(arr, pad_value=0.0):
    squeezed = False

    if arr.ndim == 2:
        arr = arr[:, :, None]
        squeezed = True

    h, w, c = arr.shape

    if h == TARGET_SIZE and w == TARGET_SIZE:
        return arr[:, :, 0] if squeezed else arr

    pad_h = TARGET_SIZE - h
    pad_w = TARGET_SIZE - w

    if pad_h < 0 or pad_w < 0:
        raise ValueError(f"Input larger than {TARGET_SIZE}: {arr.shape}")

    top = pad_h // 2
    bottom = pad_h - top
    left = pad_w // 2
    right = pad_w - left

    padded = np.pad(
        arr,
        ((top, bottom), (left, right), (0, 0)),
        mode="constant",
        constant_values=pad_value,
    )

    return padded[:, :, 0] if squeezed else padded


def pad_lonlat_to_300(lonlat):
    lon = lonlat[:, :, 0]
    lat = lonlat[:, :, 1]

    lon_pad = pad_array_to_300(lon, pad_value=0.0)
    lat_pad = pad_array_to_300(lat, pad_value=0.0)

    return np.stack([lon_pad, lat_pad], axis=-1).astype(np.float32)


def save_300_for_date(
    date_str,
    VIIRS_OUT,
    OISST_OUT,
    LAND_OUT,
    LONLAT_OUT,
    DST_ROOT,
):
    file_name = f"{date_str}.npy"

    dst_viirs = DST_ROOT / "cloud" / "viirs"
    dst_ostia = DST_ROOT / "cloud" / "oisst"
    dst_land = DST_ROOT / "cloud" / "land_mask"
    dst_lonlat = DST_ROOT / "cloud" / "lon_lat" / "viirs"

    for p in [dst_viirs, dst_ostia, dst_land, dst_lonlat]:
        p.mkdir(parents=True, exist_ok=True)

    viirs = np.load(VIIRS_OUT / file_name)
    ostia = np.load(OISST_OUT / file_name)
    land = np.load(LAND_OUT / file_name)
    lonlat = np.load(LONLAT_OUT / file_name)

    viirs_300 = pad_array_to_300(viirs, pad_value=0.0)
    ostia_300 = pad_array_to_300(ostia, pad_value=0.0)
    land_300 = pad_array_to_300(land, pad_value=1.0)
    lonlat_300 = pad_lonlat_to_300(lonlat)

    np.save(dst_viirs / file_name, viirs_300.astype(np.float32))
    np.save(dst_ostia / file_name, ostia_300.astype(np.float32))
    np.save(dst_land / file_name, land_300.astype(np.float32))
    np.save(dst_lonlat / file_name, lonlat_300.astype(np.float32))

    print(f"✅ Saved 300x300 files for {date_str}")

    return viirs_300, ostia_300, land_300


# ==========================================
# 6. PLOT CHECK
# ==========================================
def plot_check(date_str, viirs_300, ostia_300, land_300):
    viirs_vis = np.where(land_300 == 1, np.nan, viirs_300)
    ostia_vis = np.where(land_300 == 1, np.nan, ostia_300)

    vals = np.concatenate([
        viirs_vis[np.isfinite(viirs_vis)].ravel(),
        ostia_vis[np.isfinite(ostia_vis)].ravel(),
    ])

    vmin = np.nanpercentile(vals, 2)
    vmax = np.nanpercentile(vals, 98)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)

    im0 = axes[0].imshow(
        viirs_vis,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        origin="lower",
    )
    axes[0].set_title(f"VIIRS {VIIRS_PRODUCT}\n{date_str}")
    axes[0].axis("off")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(
        ostia_vis,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        origin="lower",
    )
    axes[1].set_title("OSTIA 300")
    axes[1].axis("off")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(
        land_300,
        cmap="gray",
        origin="lower",
    )
    axes[2].set_title("Land Mask 300\nwhite = land/invalid")
    axes[2].axis("off")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


# ==========================================
# 7. EXECUTION: LOOP YEARS + DAYS
# ==========================================
if __name__ == "__main__":

    all_success = {}
    all_failed = {}

    for YEAR in YEARS:

        print("\n" + "=" * 80)
        print(f"START PROCESSING YEAR {YEAR}")
        print("=" * 80)

        BASE_ROOT = Path(f"../../{SITE}/result/ready_whole_{YEAR}")
        BASE_DATA_DIR = BASE_ROOT / "cloud"

        VIIRS_OUT = BASE_DATA_DIR / "viirs"
        OISST_OUT = BASE_DATA_DIR / "oisst"
        LAND_OUT = BASE_DATA_DIR / "land_mask"
        LONLAT_OUT = BASE_DATA_DIR / "lon_lat" / "viirs"

        for p in [VIIRS_OUT, OISST_OUT, LAND_OUT, LONLAT_OUT]:
            p.mkdir(parents=True, exist_ok=True)

        DST_ROOT = Path(f"{BASE_ROOT.as_posix()}_300")
        DST_ROOT.mkdir(parents=True, exist_ok=True)

        DATES = generate_all_dates(YEAR)

        success_dates = []
        failed_dates = []

        print("\n==============================")
        print(f"YEAR: {YEAR}")
        print(f"SITE: {SITE}")
        print(f"Processing ALL valid dates")
        print(f"VIIRS_PRODUCT: {VIIRS_PRODUCT}")
        print(f"AOI center: lon={CENTER_LON}, lat={CENTER_LAT}")
        print(f"BASE_ROOT: {BASE_ROOT}")
        print(f"DST_ROOT: {DST_ROOT}")
        print("==============================")

        for target_date in tqdm(DATES, desc=f"Processing all valid days in {YEAR}"):

            viirs_file_path = find_l3_path(target_date)
            ostia_file_path = find_ostia_path(target_date)

            if not viirs_file_path:
                print(f"[MISSING VIIRS] {target_date}")
                failed_dates.append((target_date, "missing_viirs"))
                continue

            if not ostia_file_path.exists():
                print(f"[MISSING OSTIA] {target_date}")
                failed_dates.append((target_date, "missing_ostia"))
                continue

            try:
                generated_date = process_single_day_with_landmask(
                    target_date,
                    viirs_file_path,
                    ostia_file_path,
                    VIIRS_OUT,
                    OISST_OUT,
                    LAND_OUT,
                    LONLAT_OUT,
                )

                viirs_300, ostia_300, land_300 = save_300_for_date(
                    generated_date,
                    VIIRS_OUT,
                    OISST_OUT,
                    LAND_OUT,
                    LONLAT_OUT,
                    DST_ROOT,
                )

                if len(success_dates) < N_DEBUG_PLOTS:
                    plot_check(
                        generated_date,
                        viirs_300,
                        ostia_300,
                        land_300,
                    )

                success_dates.append(generated_date)

            except Exception as e:
                print(f"[FAILED] {target_date}: {e}")
                failed_dates.append((target_date, str(e)))
                continue

        print("\n===================================")
        print(f"YEAR {YEAR} SUMMARY")
        print("===================================")
        print("VIIRS_PRODUCT:", VIIRS_PRODUCT)
        print("Total successful dates:", len(success_dates))
        print("Total failed dates:", len(failed_dates))

        if success_dates:
            print("First 10 successful dates:", success_dates[:10])
            print("Last 10 successful dates:", success_dates[-10:])

        if failed_dates:
            print("First 10 failed dates:", failed_dates[:10])

        success_txt = DST_ROOT / f"successful_dates_{YEAR}_{PRODUCT_TAG}.txt"
        failed_txt = DST_ROOT / f"failed_dates_{YEAR}_{PRODUCT_TAG}.txt"

        with open(success_txt, "w") as f:
            for d in success_dates:
                f.write(f"{d}\n")

        with open(failed_txt, "w") as f:
            for d, reason in failed_dates:
                f.write(f"{d} : {reason}\n")

        print("\nSaved:")
        print(success_txt)
        print(failed_txt)

        all_success[YEAR] = success_dates
        all_failed[YEAR] = failed_dates

    print("\n" + "=" * 80)
    print("FINAL ALL-YEAR SUMMARY")
    print("=" * 80)

    for YEAR in YEARS:
        print(
            f"{YEAR}: "
            f"success = {len(all_success[YEAR])}, "
            f"failed = {len(all_failed[YEAR])}"
        )

    print("\n✅ ALL YEARS DONE.")